# Omo Forest Reserve Ecological Informatics Project

# Notebook 05

## Community Matrix Construction, Validation and Data Engineering

---

### Project

Ecological Informatics Assessment of Tree Communities in Omo Forest Reserve, Nigeria

---

### Purpose

This notebook reconstructs the original tree inventory field tables into a standardized, validated community matrix suitable for multivariate ecological analyses.

Unlike the previous notebooks, this notebook focuses exclusively on ecological data engineering and quality assurance.

No ecological analyses are performed here.

The outputs generated in this notebook become the primary input datasets for Notebook 06.

---

### Inputs

• Original Word Document

Species_abundance_and_distribution_of_the_Riparian_forest.docx

---

### Outputs

• community_matrix_90plots.csv

• plot_metadata.csv

• species_metadata.csv

• data_integrity_report.csv

• community_matrix.xlsx

---

# Table of Contents

1. Import Libraries

2. Project Configuration

3. Load Source Document

4. Inspect Source Document

5. Extract Tree Inventory Tables

6. Standardize Tree Tables

7. Build Community Matrix

8. Validate Community Matrix

9. Export Outputs

10. Notebook Summary

In [155]:
# =============================================================================
# Import Libraries
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from docx import Document

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
sns.set_context("talk")

print("✓ Libraries imported successfully.")

✓ Libraries imported successfully.


In [156]:
# =============================================================================
# Project Configuration
# =============================================================================

project_root = Path.cwd().parent

PROJECT = "Omo Forest Ecological Informatics Project"

WORD_FILE = project_root / "Species_abundance_and_distribution_of_the_Riparian_forest.docx"

OUTPUT_DIR = project_root / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT)

print()

print("Output Folder")

print(OUTPUT_DIR.resolve())

Omo Forest Ecological Informatics Project

Output Folder
C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\outputs


In [157]:
# =============================================================================
# Load Source Document
# =============================================================================

assert WORD_FILE.exists(), f"{WORD_FILE} not found."

document = Document(WORD_FILE)

print("="*70)

print("SOURCE DOCUMENT")

print("="*70)

print(f"Paragraphs : {len(document.paragraphs)}")

print(f"Tables     : {len(document.tables)}")

SOURCE DOCUMENT
Paragraphs : 136
Tables     : 21


In [158]:
# =============================================================================
# Inspect Document Tables
# =============================================================================

table_inventory = []

for i, table in enumerate(document.tables):

    table_inventory.append({

        "Table": i + 1,

        "Rows": len(table.rows),

        "Columns": len(table.columns),

        "Header":
        " | ".join(
            cell.text.strip()
            for cell in table.rows[0].cells[:5]
        )

    })

table_inventory = pd.DataFrame(table_inventory)

display(table_inventory)

,Table,Rows,Columns,Header
0,1,28,18,SPECIES | CmP1 | CmP2 | CmP3 | CmP4
1,2,26,17,Species | CSP1 | CsP2 | CsP3 | CsP4
2,3,26,17,Species | CuP1 | CuP2 | CuP3 | CuP4
3,4,27,17,SPECIES | BmP1 | BmP2 | BmP3 | BmP4
4,5,35,17,Species | BsP1 | BsP2 | BsP3 | BsP4
5,6,30,17,SPECIES | BuP1 | BuP2 | BuP3 | BuP4
6,7,29,17,SPECIES | TmP1 | TmP2 | TmP3 | TmP4
7,8,16,17,SPECIES | TsP1 | TsP2 | TsP3 | TsP4
8,9,20,17,Species | TuP1 | TuP2 | TuP3 | TuP4
9,10,88,4,SPECIES | FAMILY | OCCURENCE |


# 5. Extract Tree Inventory Tables

The first nine tables of the source document contain the tree inventory data collected from the nine ecological communities within Omo Forest Reserve.

These tables are extracted without modification and stored as independent pandas DataFrames for subsequent validation and transformation.

In [159]:
# =============================================================================
# Ecological Community Metadata
# =============================================================================

COMMUNITIES = {
    "Table_1": {"Code": "CM", "Zone": "Core", "Habitat": "Major River"},
    "Table_2": {"Code": "CS", "Zone": "Core", "Habitat": "Stream"},
    "Table_3": {"Code": "CU", "Zone": "Core", "Habitat": "Upland"},
    "Table_4": {"Code": "BM", "Zone": "Buffer", "Habitat": "Major River"},
    "Table_5": {"Code": "BS", "Zone": "Buffer", "Habitat": "Stream"},
    "Table_6": {"Code": "BU", "Zone": "Buffer", "Habitat": "Upland"},
    "Table_7": {"Code": "TM", "Zone": "Transition", "Habitat": "Major River"},
    "Table_8": {"Code": "TS", "Zone": "Transition", "Habitat": "Stream"},
    "Table_9": {"Code": "TU", "Zone": "Transition", "Habitat": "Upland"},
}

pd.DataFrame(COMMUNITIES).T

,Code,Zone,Habitat
Table_1,CM,Core,Major River
Table_2,CS,Core,Stream
Table_3,CU,Core,Upland
Table_4,BM,Buffer,Major River
Table_5,BS,Buffer,Stream
Table_6,BU,Buffer,Upland
Table_7,TM,Transition,Major River
Table_8,TS,Transition,Stream
Table_9,TU,Transition,Upland


In [160]:
# =============================================================================
# Convert Word Table to DataFrame
# =============================================================================

def word_table_to_dataframe(table):
    """
    Convert a Word table into a pandas DataFrame.
    """

    rows = []

    for row in table.rows:
        rows.append([cell.text.strip() for cell in row.cells])

    df = pd.DataFrame(rows)

    # First row becomes header
    df.columns = df.iloc[0]

    # Remove header row
    df = df.iloc[1:].reset_index(drop=True)

    return df

In [161]:
# =============================================================================
# Extract Tree Inventory Tables
# =============================================================================

tree_tables = {}

for i in range(9):

    table_name = f"Table_{i+1}"

    tree_tables[table_name] = word_table_to_dataframe(
        document.tables[i]
    )

print(f"{len(tree_tables)} tree inventory tables extracted.")

9 tree inventory tables extracted.


In [162]:
# =============================================================================
# Tree Inventory Table Summary
# =============================================================================

summary = []

for name, table in tree_tables.items():

    summary.append({

        "Community": name,

        "Rows": table.shape[0],

        "Columns": table.shape[1],

        "First Column": table.columns[0],

        "Last Column": table.columns[-1]

    })

summary = pd.DataFrame(summary)

display(summary)

,Community,Rows,Columns,First Column,Last Column
0,Table_1,27,18,SPECIES,RIV
1,Table_2,25,17,Species,RIV
2,Table_3,25,17,Species,RIV
3,Table_4,26,17,SPECIES,RIV
4,Table_5,34,17,Species,RIV
5,Table_6,29,17,SPECIES,RIV
6,Table_7,28,17,SPECIES,RIV
7,Table_8,15,17,SPECIES,RIV
8,Table_9,19,17,Species,RIV


In [163]:
# =============================================================================
# Preview Table 8 (Transition Stream)
# =============================================================================

display(tree_tables["Table_8"])

,SPECIES,TsP1,TsP2,TsP3,TsP4,TsP5,TsP5,TsP7,TsP8,TsP9,TsP 10,Frequency,Individual,density,Rel density,Rel. Freq.,RIV
0,Albizia zygia,,,1,,,,1,,,,2,2,0.0027,0.9569,3.2258,2.0914
1,Alstonei bonei,,1,1,4,,1,4,3,,4,7,18,0.0248,8.6124,11.290,9.9514
2,Anthocliesta vogelii,1,,1,14,12,,7,12,1,3,8,51,0.0703,24.401,12.903,18.653
3,Bombax buonopozense,Bombax buonopozense,1,,,,1,,,11,,3,13,0.0179,6.2201,4.8387,5.5294
4,Bridelia ferruginea,1,,2,,,2,1,,,1,5,7,0.0096,3.3492,8.0645,5.7069
5,Cleistoformis patens,4,2,,7,11,4,5,9,4,3,9,49,0.0675,23.445,14.516,18.981
6,Elaeis guinensis,2,4,,,,4,,,4,,4,14,0.0193,6.6985,6.4516,6.5751
7,Ficus mucosa,6,3,1,,,5,,,8,1,6,24,0.0331,11.483,9.6774,10.58
8,Gmelina arborea,,,2,1,1,,2,2,,1,6,9,0.0124,4.3062,9.6774,6.9918
9,Harungana madagascariensis,Harungana madagascariensis,,,,,1,,,,1,2,2,0.0027,0.9569,3.2258,2.0914


# 6. Data Validation

Before any transformation is performed, the extracted community tables are subjected to a series of validation checks.

These checks are designed to identify potential issues in the original field tables, including:

- Duplicate plot identifiers
- Incorrect plot numbering
- Invalid abundance values
- Missing species names
- Duplicate species records within a table
- Unexpected table dimensions

No corrections are performed at this stage. The objective is to document the integrity of the source data before transformation.

In [164]:
# =============================================================================
# Validate Community Tables
# =============================================================================

def validate_tree_table(table_name, df):

    report = {}

    report["Community"] = table_name

    report["Rows"] = df.shape[0]

    report["Columns"] = df.shape[1]

    report["Duplicate Columns"] = int(pd.Index(df.columns).duplicated().sum())

    report["Blank Species"] = int(
        df.iloc[:,0].astype(str).str.strip().eq("").sum()
    )

    report["Duplicate Species"] = int(
        df.iloc[:,0].duplicated().sum()
    )

    return report

In [165]:
# =============================================================================
# Validate All Community Tables
# =============================================================================

validation = []

for table_name, df in tree_tables.items():

    validation.append(
        validate_tree_table(table_name, df)
    )

validation = pd.DataFrame(validation)

display(validation)

,Community,Rows,Columns,Duplicate Columns,Blank Species,Duplicate Species
0,Table_1,27,18,0,0,0
1,Table_2,25,17,0,0,0
2,Table_3,25,17,0,0,0
3,Table_4,26,17,0,0,0
4,Table_5,34,17,0,0,0
5,Table_6,29,17,0,0,0
6,Table_7,28,17,0,0,0
7,Table_8,15,17,1,0,0
8,Table_9,19,17,0,0,0


In [166]:
# =============================================================================
# Inspect Plot Headers
# =============================================================================

header_report = []

for table_name, df in tree_tables.items():

    headers = list(df.columns[1:11])

    header_report.append({

        "Community": table_name,

        "Plot Headers": headers

    })

header_report = pd.DataFrame(header_report)

display(header_report)

,Community,Plot Headers
0,Table_1,"[CmP1, CmP2, CmP3, CmP4, CmP5, CmP6, CmP7, CmP..."
1,Table_2,"[CSP1, CsP2, CsP3, CsP4, CsP5, CsP6, CsP7, CsP..."
2,Table_3,"[CuP1, CuP2, CuP3, CuP4, CuP5, CuP6, CuP7, CuP..."
3,Table_4,"[BmP1, BmP2, BmP3, BmP4, BmP5, BmP6, BmP7, BmP..."
4,Table_5,"[BsP1, BsP2, BsP3, BsP4, BsP5, BsP6, BsP7, BsP..."
5,Table_6,"[BuP1, BuP2, BuP3, BuP4, BuP5, BuP6, BuP7, BuP..."
6,Table_7,"[TmP1, TmP2, TmP3, TmP4, TmP5, TmP6, TmP7, TmP..."
7,Table_8,"[TsP1, TsP2, TsP3, TsP4, TsP5, TsP5, TsP7, TsP..."
8,Table_9,"[TuP1, TuP2, TuP3, TuP4, TuP5, TuP6, TuP7, TuP..."


In [167]:
# =============================================================================
# Detect Invalid Plot Headers
# =============================================================================

for table_name, df in tree_tables.items():

    print("="*70)

    print(table_name)

    print("="*70)

    headers = list(df.columns[1:11])

    duplicates = pd.Index(headers).duplicated()

    if duplicates.any():

        print("Duplicate headers detected")

        print(pd.Index(headers)[duplicates])

    else:

        print("No duplicate headers.")

    print()

Table_1
No duplicate headers.

Table_2
No duplicate headers.

Table_3
No duplicate headers.

Table_4
No duplicate headers.

Table_5
No duplicate headers.

Table_6
No duplicate headers.

Table_7
No duplicate headers.

Table_8
Duplicate headers detected
Index(['TsP5'], dtype='object')

Table_9
No duplicate headers.



# 7. Data Transformation

Following successful validation, the extracted community tables are transformed into a standardized format suitable for ecological analyses.

The transformation process includes:

- Standardizing species names
- Standardizing plot identifiers
- Cleaning abundance values
- Preparing community matrices

Each transformation is deterministic and documented to ensure full reproducibility.

In [168]:
# =============================================================================
# Standardize Tree Tables
# =============================================================================

clean_tables = {}

for table_name, df in tree_tables.items():

    table = df.copy()

    # Remove leading/trailing spaces from column names
    table.columns = [str(c).strip() for c in table.columns]

    # Standardize first column
    cols = list(table.columns)
    cols[0] = "Species"
    table.columns = cols

    # Standardize species names
    table["Species"] = (
        table["Species"]
        .astype(str)
        .str.strip()
    )

    clean_tables[table_name] = table

print(f"{len(clean_tables)} tables standardized.")

9 tables standardized.


In [169]:
# =============================================================================
# Standardize Plot Headers
# =============================================================================

for table_name, table in clean_tables.items():

    code = COMMUNITIES[table_name]["Code"]

    # Preserve original style:
    # CM → Cm
    # TS → Ts

    prefix = code[0] + code[1].lower()

    cols = list(table.columns)

    # Force the first ten abundance columns
    # to be P1 ... P10

    for i in range(10):

        cols[i + 1] = f"{prefix}P{i + 1}"

    table.columns = cols

    clean_tables[table_name] = table

print("✓ Plot headers standardized.")

✓ Plot headers standardized.


In [170]:
# =============================================================================
# Verify Plot Headers
# =============================================================================

for table_name, table in clean_tables.items():

    print("="*60)

    print(table_name)

    print("="*60)

    print(list(table.columns[:11]))

    print()

Table_1
['Species', 'CmP1', 'CmP2', 'CmP3', 'CmP4', 'CmP5', 'CmP6', 'CmP7', 'CmP8', 'CmP9', 'CmP10']

Table_2
['Species', 'CsP1', 'CsP2', 'CsP3', 'CsP4', 'CsP5', 'CsP6', 'CsP7', 'CsP8', 'CsP9', 'CsP10']

Table_3
['Species', 'CuP1', 'CuP2', 'CuP3', 'CuP4', 'CuP5', 'CuP6', 'CuP7', 'CuP8', 'CuP9', 'CuP10']

Table_4
['Species', 'BmP1', 'BmP2', 'BmP3', 'BmP4', 'BmP5', 'BmP6', 'BmP7', 'BmP8', 'BmP9', 'BmP10']

Table_5
['Species', 'BsP1', 'BsP2', 'BsP3', 'BsP4', 'BsP5', 'BsP6', 'BsP7', 'BsP8', 'BsP9', 'BsP10']

Table_6
['Species', 'BuP1', 'BuP2', 'BuP3', 'BuP4', 'BuP5', 'BuP6', 'BuP7', 'BuP8', 'BuP9', 'BuP10']

Table_7
['Species', 'TmP1', 'TmP2', 'TmP3', 'TmP4', 'TmP5', 'TmP6', 'TmP7', 'TmP8', 'TmP9', 'TmP10']

Table_8
['Species', 'TsP1', 'TsP2', 'TsP3', 'TsP4', 'TsP5', 'TsP6', 'TsP7', 'TsP8', 'TsP9', 'TsP10']

Table_9
['Species', 'TuP1', 'TuP2', 'TuP3', 'TuP4', 'TuP5', 'TuP6', 'TuP7', 'TuP8', 'TuP9', 'TuP10']



In [171]:
# =============================================================================
# Clean Abundance Values
# =============================================================================

clean_tables_numeric = {}

for table_name, table in clean_tables.items():

    df = table.copy()

    abundance_cols = list(df.columns[1:11])

    # Convert abundance values to numeric
    df[abundance_cols] = (
        df[abundance_cols]
        .replace("", 0)
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype(int)
    )

    clean_tables_numeric[table_name] = df

print("✓ Abundance values converted to numeric.")

✓ Abundance values converted to numeric.


In [172]:
# =============================================================================
# Validate Abundance Columns
# =============================================================================

for table_name, table in clean_tables_numeric.items():

    abundance_cols = table.columns[1:11]

    print("="*60)

    print(table_name)

    print("="*60)

    print(table[abundance_cols].dtypes)

    print()

Table_1
CmP1     int64
CmP2     int64
CmP3     int64
CmP4     int64
CmP5     int64
CmP6     int64
CmP7     int64
CmP8     int64
CmP9     int64
CmP10    int64
dtype: object

Table_2
CsP1     int64
CsP2     int64
CsP3     int64
CsP4     int64
CsP5     int64
CsP6     int64
CsP7     int64
CsP8     int64
CsP9     int64
CsP10    int64
dtype: object

Table_3
CuP1     int64
CuP2     int64
CuP3     int64
CuP4     int64
CuP5     int64
CuP6     int64
CuP7     int64
CuP8     int64
CuP9     int64
CuP10    int64
dtype: object

Table_4
BmP1     int64
BmP2     int64
BmP3     int64
BmP4     int64
BmP5     int64
BmP6     int64
BmP7     int64
BmP8     int64
BmP9     int64
BmP10    int64
dtype: object

Table_5
BsP1     int64
BsP2     int64
BsP3     int64
BsP4     int64
BsP5     int64
BsP6     int64
BsP7     int64
BsP8     int64
BsP9     int64
BsP10    int64
dtype: object

Table_6
BuP1     int64
BuP2     int64
BuP3     int64
BuP4     int64
BuP5     int64
BuP6     int64
BuP7     int64
BuP8     int64
BuP9   

In [173]:
# =============================================================================
# Verify Invalid Abundance Correction
# =============================================================================

bombax = clean_tables_numeric["Table_8"]

bombax = bombax[
    bombax["Species"] == "Bombax buonopozense"
]

display(bombax)

,Species,TsP1,TsP2,TsP3,TsP4,TsP5,TsP6,TsP7,TsP8,TsP9,TsP10,Frequency,Individual,density,Rel density,Rel. Freq.,RIV
3,Bombax buonopozense,0,1,0,0,0,1,0,0,11,0,3,13,0.0179,6.2201,4.8387,5.5294


In [174]:
# =============================================================================
# Community Integrity Summary
# =============================================================================

integrity = []

for table_name, table in clean_tables_numeric.items():

    abundance_cols = table.columns[1:11]

    integrity.append({

        "Community": table_name,

        "Species": len(table),

        "Plots": len(abundance_cols),

        "Total Individuals":
        table[abundance_cols].sum().sum()

    })

integrity = pd.DataFrame(integrity)

display(integrity)

,Community,Species,Plots,Total Individuals
0,Table_1,27,10,267
1,Table_2,25,10,183
2,Table_3,25,10,230
3,Table_4,26,10,180
4,Table_5,34,10,225
5,Table_6,29,10,229
6,Table_7,28,10,137
7,Table_8,15,10,209
8,Table_9,19,10,90


# 8. Community Matrix Construction

Each ecological community table is transformed into a standard
**plot × species abundance matrix**.

During this transformation:

- Plot identifiers become rows.
- Species become columns.
- Abundance values populate the matrix.
- Ecological metadata (Zone and Habitat) are attached to each plot.

The resulting community matrices are merged into a single master community matrix representing all sampling plots in Omo Forest Reserve.

In [175]:
# =============================================================================
# Build Community Matrix
# =============================================================================

def build_community_matrix(table_name, table):

    abundance_cols = list(table.columns[1:11])

    species = table["Species"]

    abundance = table[abundance_cols].copy()

    # Species become column names
    abundance.index = species

    community = abundance.T

    community.index.name = "Plot_ID"

    community.reset_index(inplace=True)

    # Attach ecological metadata
    community.insert(
        1,
        "Zone",
        COMMUNITIES[table_name]["Zone"]
    )

    community.insert(
        2,
        "Habitat",
        COMMUNITIES[table_name]["Habitat"]
    )

    return community

In [178]:
# =============================================================================
# Test Community Matrix
# =============================================================================

cm = build_community_matrix(
    "Table_1",
    clean_tables_numeric["Table_1"]
)

display(cm.head())

print()

print(cm.shape)

Species,Plot_ID,Zone,Habitat,Amphimas tetracoides,Annonidium manni,Anthonotha macrophylla,Blighia Sapida,Celtis zenkeri,Cleistopholis patens,Diospyros dendo,...,Picralima nitida,Pycnanthus angolensis,Ricinodendron heudelotii,Scotellia coricea,Sterculia rhinopetala,Strombosia pustulata,Trichilia gilgiana,Trichilia monadelpha,Uapaca heudelotii,26 species
0,CmP1,Core,Major River,1,2,0,0,4,1,3,...,0,2,0,0,0,3,0,0,0,0
1,CmP2,Core,Major River,0,2,0,1,0,0,1,...,7,1,1,0,1,6,1,0,0,0
2,CmP3,Core,Major River,0,1,1,0,0,0,2,...,4,0,2,1,1,4,0,0,0,0
3,CmP4,Core,Major River,0,0,0,0,0,1,5,...,2,0,0,0,3,2,1,0,0,0
4,CmP5,Core,Major River,0,0,0,0,0,0,7,...,6,0,0,0,1,6,0,0,0,0



(10, 30)


In [179]:
# =============================================================================
# Build Community Matrices for All Ecological Communities
# =============================================================================

community_matrices = {}

for table_name, table in clean_tables_numeric.items():

    community = build_community_matrix(table_name, table)

    community_matrices[table_name] = community

    print(
        f"{table_name:<8}"
        f" -> {community.shape[0]} plots × {community.shape[1]-3} columns"
    )

print("\n✓ All community matrices successfully created.")

Table_1  -> 10 plots × 27 columns
Table_2  -> 10 plots × 25 columns
Table_3  -> 10 plots × 25 columns
Table_4  -> 10 plots × 26 columns
Table_5  -> 10 plots × 34 columns
Table_6  -> 10 plots × 29 columns
Table_7  -> 10 plots × 28 columns
Table_8  -> 10 plots × 15 columns
Table_9  -> 10 plots × 19 columns

✓ All community matrices successfully created.


In [180]:
# =============================================================================
# Merge Community Matrices
# =============================================================================

community_matrix = pd.concat(

    community_matrices.values(),

    ignore_index=True,

    sort=False

)

print("=" * 70)
print("MASTER COMMUNITY MATRIX CREATED")
print("=" * 70)

print(f"Plots   : {community_matrix.shape[0]}")
print(f"Columns : {community_matrix.shape[1]}")

MASTER COMMUNITY MATRIX CREATED
Plots   : 90
Columns : 151


In [181]:
# =============================================================================
# Replace Missing Species Abundances
# =============================================================================

metadata_cols = ["Plot_ID", "Zone", "Habitat"]

species_cols = [

    c

    for c in community_matrix.columns

    if c not in metadata_cols

]

community_matrix[species_cols] = (

    community_matrix[species_cols]

    .fillna(0)

    .astype(int)

)

print("✓ Missing abundances replaced with zero.")

✓ Missing abundances replaced with zero.


In [182]:
# =============================================================================
# Community Matrix Validation
# =============================================================================

print("=" * 70)
print("MASTER COMMUNITY MATRIX VALIDATION")
print("=" * 70)

print(f"Number of plots          : {len(community_matrix)}")

print(f"Unique species           : {len(species_cols)}")

print(f"Duplicate Plot IDs       : {community_matrix['Plot_ID'].duplicated().sum()}")

print(f"Missing abundance values : {community_matrix[species_cols].isna().sum().sum()}")

print(f"Total individuals        : {community_matrix[species_cols].sum().sum():,}")

MASTER COMMUNITY MATRIX VALIDATION
Number of plots          : 90
Unique species           : 148
Duplicate Plot IDs       : 0
Missing abundance values : 0
Total individuals        : 1,750


In [183]:
# =============================================================================
# Community Summary
# =============================================================================

summary = (

    community_matrix

    .groupby(["Zone", "Habitat"])

    .agg(

        Number_of_Plots=("Plot_ID", "count")

    )

    .reset_index()

)

display(summary)

,Zone,Habitat,Number_of_Plots
0,Buffer,Major River,10
1,Buffer,Stream,10
2,Buffer,Upland,10
3,Core,Major River,10
4,Core,Stream,10
5,Core,Upland,10
6,Transition,Major River,10
7,Transition,Stream,10
8,Transition,Upland,10


# 9. Export Outputs

The validated community matrix and associated metadata are exported for subsequent ecological analyses.

These outputs serve as the primary inputs to Notebook 06.

In [184]:
# =============================================================================
# Export Master Community Matrix
# =============================================================================

community_matrix.to_csv(

    TABLE_DIR / "community_matrix_90plots.csv",

    index=False

)

community_matrix.to_excel(

    TABLE_DIR / "community_matrix_90plots.xlsx",

    index=False

)

print("✓ Master community matrix exported.")

✓ Master community matrix exported.


In [185]:
# =============================================================================
# Export Plot Metadata
# =============================================================================

plot_metadata = community_matrix[
    ["Plot_ID", "Zone", "Habitat"]
].copy()

plot_metadata.to_csv(

    TABLE_DIR / "plot_metadata.csv",

    index=False

)

display(plot_metadata.head())

Species,Plot_ID,Zone,Habitat
0,CmP1,Core,Major River
1,CmP2,Core,Major River
2,CmP3,Core,Major River
3,CmP4,Core,Major River
4,CmP5,Core,Major River


In [186]:
# =============================================================================
# Export Species Metadata
# =============================================================================

species_metadata = pd.DataFrame({

    "Species": species_cols

})

species_metadata.to_csv(

    TABLE_DIR / "species_metadata.csv",

    index=False

)

display(species_metadata.head())

,Species
0,Amphimas tetracoides
1,Annonidium manni
2,Anthonotha macrophylla
3,Blighia Sapida
4,Celtis zenkeri


In [187]:
# =============================================================================
# Export Data Integrity Report
# =============================================================================

integrity_report = pd.DataFrame({

    "Metric": [

        "Communities",

        "Plots",

        "Unique Species",

        "Duplicate Plot IDs",

        "Missing Abundance Values",

        "Total Individuals"

    ],

    "Value": [

        9,

        len(community_matrix),

        len(species_cols),

        community_matrix["Plot_ID"].duplicated().sum(),

        community_matrix[species_cols].isna().sum().sum(),

        community_matrix[species_cols].sum().sum()

    ]

})

integrity_report.to_csv(

    TABLE_DIR / "data_integrity_report.csv",

    index=False

)

display(integrity_report)

,Metric,Value
0,Communities,9
1,Plots,90
2,Unique Species,148
3,Duplicate Plot IDs,0
4,Missing Abundance Values,0
5,Total Individuals,1750


# 10. Notebook Summary

## Objectives Achieved

✔ Extracted the original tree inventory tables from the source document.

✔ Validated the structural integrity of all community tables.

✔ Corrected plot identifier inconsistencies through deterministic reconstruction.

✔ Cleaned abundance values and standardized species records.

✔ Constructed nine community matrices.

✔ Merged all communities into a single master community matrix.

✔ Produced a validated dataset comprising:

- 90 sampling plots

- 148 unique tree species

- 1,750 recorded individuals

- Zero duplicate plot identifiers

- Zero missing abundance values

## Outputs Generated

- community_matrix_90plots.csv

- community_matrix_90plots.xlsx

- plot_metadata.csv

- species_metadata.csv

- data_integrity_report.csv

---

The exported datasets provide the standardized input required for Notebook 06, where multivariate community ecology analyses will be undertaken, including alpha diversity, beta diversity, community similarity, ordination, and hypothesis testing.